# MiroFish-Offline — Kaggle Ollama Server (with saved dataset)

This notebook loads models from your saved Kaggle dataset (no re-downloading).
Then exposes Ollama via **ngrok** for your local MiroFish to use.

### BEFORE RUNNING
- Run `mirofish_save_model_to_dataset.ipynb` **once first** to create the dataset
- Enable **GPU T4 x2**: Notebook Settings → Accelerator
- Add your dataset as input: **File → Add Input → Datasets → search `ollama-qwen25-14b-nomic`**
- Add secret `NGROK_TOKEN`: Add-ons → Secrets
  - Get free token at https://dashboard.ngrok.com/get-started/your-authtoken

In [12]:
# Check GPU
!nvidia-smi

Mon May 11 07:29:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
# Step 1: Install Ollama Engine
# This installs the Ollama program (engine/tool only — no models downloaded here)
# Models are restored separately from the Kaggle dataset in the next step.
#
# zstd: compression tool required by Ollama's installer
#       (Ollama downloads as .tar.zst format, not pre-installed on Kaggle/Ubuntu)
#
# Note: "Unable to detect NVIDIA/AMD GPU" warning is expected — ignore it.
#       GPU becomes available once a model is actually loaded and run.

import os
os.environ["PATH"] += ":/usr/local/bin"

print("Installing zstd...")
!apt-get install -y zstd -q

print("Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh
print("Done.")

Installing zstd...
Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
Installing Ollama...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Done.


In [14]:
# Step 2: Restore Models from Kaggle Dataset

# The dataset contains pre-downloaded model weights/blobs saved earlier.
# This avoids re-downloading models (~9GB) every time the notebook runs.

# What this does:
#   1. Checks the dataset is mounted (File → Add Input → Datasets)
#   2. Wipes any existing ~/.ollama/models directory (clean restore)
#   3. Copies model blobs from dataset → ~/.ollama/models

# Models restored:
#   - qwen2.5:14b       (~9GB) — main LLM for ontology/graph generation
#   - nomic-embed-text  (~274MB) — embedding model


import shutil, os

DATASET_PATH = "/kaggle/input/datasets/ananyasrinivas/ollama-qwen25-14b-nomic" 
OLLAMA_DIR   = os.path.expanduser("~/.ollama/models")

if not os.path.exists(DATASET_PATH):
    raise RuntimeError(
        "Dataset not found! Add it via File → Add Input → Datasets → ollama-qwen25-7b-nomic"
    )

print("Restoring model blobs from dataset...")
if os.path.exists(OLLAMA_DIR):
    shutil.rmtree(OLLAMA_DIR)
shutil.copytree(DATASET_PATH, OLLAMA_DIR)

print("Done.")

Restoring model blobs from dataset...
Done. Verifying:


In [15]:
#Step 3: Start Ollama Server in Background 
# Starts the Ollama server as a background process so it keeps running while other cells execute.
#

# Port 11434: default Ollama port
#
# Flow:
#   1. Kill any existing Ollama process (clean start, avoids port conflicts)
#   2. Start fresh Ollama server in background
#   3. Wait 10s for server to fully initialize
#   4. Verify server is up and models are visible


import subprocess, time, os

# Kill current ollama
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(3)

# Start ollama listening on all interfaces
env = os.environ.copy()
env["PATH"] += ":/usr/local/bin"
env["OLLAMA_HOST"] = "0.0.0.0:11434"

proc = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=env
)
time.sleep(10)

# Verify
import requests
r = requests.get("http://localhost:11434/api/tags", timeout=5)
print(r.json())

{'models': [{'name': 'nomic-embed-text:latest', 'model': 'nomic-embed-text:latest', 'modified_at': '2026-05-11T06:22:56.047697228Z', 'size': 274302450, 'digest': '0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'nomic-bert', 'families': ['nomic-bert'], 'parameter_size': '137M', 'quantization_level': 'F16'}}, {'name': 'qwen2.5:14b', 'model': 'qwen2.5:14b', 'modified_at': '2026-05-11T06:22:56.066965684Z', 'size': 8988124069, 'digest': '7cdf5a0187d5c58cc5d369b255592f7841d1c4696d45a8c8a9489440385b22f6', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen2', 'families': ['qwen2'], 'parameter_size': '14.8B', 'quantization_level': 'Q4_K_M'}}]}


In [16]:
#Quick smoke-test

import requests, json

resp = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": "qwen2.5:14b", "prompt": "Reply with OK only.", "stream": False}
)
print("LLM response:", resp.json().get("response", resp.text)[:80])

resp2 = requests.post(
    "http://localhost:11434/api/embeddings",
    json={"model": "nomic-embed-text", "prompt": "test"}
)
emb = resp2.json().get("embedding", [])
print(f"Embedding dim: {len(emb)}  ✓" if emb else "Embedding FAILED")

LLM response: OK
Embedding dim: 768  ✓


In [17]:
# Step 4: Expose Ollama via ngrok Tunnel
# Creates a secure public HTTPS tunnel from ngrok → Kaggle's Ollama server
# This allows your local Mac backend to reach Ollama running on Kaggle.

# Why ngrok?
#   Kaggle notebooks have no public IP — ngrok gives a public URL that
#   forwards requests through to localhost:11434 on this Kaggle instance.

# bind_tls=True  → HTTPS only (secure)
# inspect=False  → disables ngrok browser warning page that breaks API calls

# After running:
#   Copy the printed values into your local .env file on Mac
#   then restart your Flask backend to pick up the new URL.


!pip install pyngrok -q

from pyngrok import ngrok, conf
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
NGROK_TOKEN = secrets.get_secret("NGROK_TOKEN")
conf.get_default().auth_token = NGROK_TOKEN

ngrok.kill()
import time
time.sleep(2)

tunnel = ngrok.connect(
    11434, 
    "http",
    bind_tls=True,
    inspect=False
)
ngrok_url = tunnel.public_url
print(f"New URL: {ngrok_url}")
print(f"LLM_BASE_URL={ngrok_url}/v1")
print(f"EMBEDDING_BASE_URL={ngrok_url}")

New URL: https://stylus-penny-freebee.ngrok-free.dev
LLM_BASE_URL=https://stylus-penny-freebee.ngrok-free.dev/v1
EMBEDDING_BASE_URL=https://stylus-penny-freebee.ngrok-free.dev


In [18]:
#To check if ollama is alive - if there is no output then server dead

import requests
r = requests.get("http://localhost:11434/api/tags", timeout=10)
print(r.json())

{'models': [{'name': 'nomic-embed-text:latest', 'model': 'nomic-embed-text:latest', 'modified_at': '2026-05-11T06:22:56.047697228Z', 'size': 274302450, 'digest': '0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'nomic-bert', 'families': ['nomic-bert'], 'parameter_size': '137M', 'quantization_level': 'F16'}}, {'name': 'qwen2.5:14b', 'model': 'qwen2.5:14b', 'modified_at': '2026-05-11T06:22:56.066965684Z', 'size': 8988124069, 'digest': '7cdf5a0187d5c58cc5d369b255592f7841d1c4696d45a8c8a9489440385b22f6', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen2', 'families': ['qwen2'], 'parameter_size': '14.8B', 'quantization_level': 'Q4_K_M'}}]}


In [ ]:
#Keep-alive WITH auto-restart
import time, subprocess, requests, os

os.environ["PATH"] += ":/usr/local/bin"

def is_ollama_alive():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        return r.status_code == 200
    except:
        return False

def restart_ollama():
    print("  Restarting Ollama...")
    subprocess.Popen(
        ["/usr/local/bin/ollama", "serve"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    time.sleep(8)

print("Keep-alive with auto-restart running...")
while True:
    if not is_ollama_alive():
        print(f"  [{time.strftime('%H:%M:%S')}] Ollama is DOWN — restarting...")
        restart_ollama()
        print(f"  [{time.strftime('%H:%M:%S')}] Ollama restarted")
    else:
        print(f"  [{time.strftime('%H:%M:%S')}] Ollama alive ✓")
    time.sleep(20)

Keep-alive with auto-restart running...
  [07:33:46] Ollama alive ✓
  [07:34:06] Ollama alive ✓
  [07:34:26] Ollama alive ✓
  [07:34:46] Ollama alive ✓
  [07:35:06] Ollama alive ✓
  [07:35:26] Ollama alive ✓
  [07:35:46] Ollama alive ✓
  [07:36:06] Ollama alive ✓
  [07:36:26] Ollama alive ✓
  [07:36:46] Ollama alive ✓
  [07:37:06] Ollama alive ✓
  [07:37:26] Ollama alive ✓
  [07:37:46] Ollama alive ✓
  [07:38:06] Ollama alive ✓
  [07:38:26] Ollama alive ✓
  [07:38:46] Ollama alive ✓
  [07:39:06] Ollama alive ✓
  [07:39:26] Ollama alive ✓
  [07:39:46] Ollama alive ✓
  [07:40:06] Ollama alive ✓
  [07:40:26] Ollama alive ✓
  [07:40:46] Ollama alive ✓
  [07:41:06] Ollama alive ✓
  [07:41:26] Ollama alive ✓
  [07:41:46] Ollama alive ✓
  [07:42:06] Ollama alive ✓
  [07:42:26] Ollama alive ✓
  [07:42:46] Ollama alive ✓
  [07:43:06] Ollama alive ✓
  [07:43:26] Ollama alive ✓
  [07:43:46] Ollama alive ✓
  [07:44:06] Ollama alive ✓
  [07:44:26] Ollama alive ✓
  [07:44:46] Ollama alive ✓
  [07:45